In [4]:
import pandas as pd
from datetime import datetime, timedelta
import requests
from io import StringIO

SP500_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
SP400_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies"


def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Wikipedia tables sometimes have MultiIndex columns like:
    ('Added', 'Ticker'), ('Removed', 'Ticker')
    This flattens them into:
    'Added_Ticker', 'Removed_Ticker'
    """
    df = df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x) for x in col if str(x) != "nan"]).strip()
            for col in df.columns
        ]
    else:
        df.columns = [str(c).strip() for c in df.columns]

    return df

def read_wikipedia_tables(url: str) -> list[pd.DataFrame]:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    return pd.read_html(StringIO(response.text))
def find_change_table(url: str) -> pd.DataFrame:
    """
    Fetch all tables from a Wikipedia index page and return the table
    that looks like the index-change table.
    """
    import requests
    from io import StringIO
    

    tables = read_wikipedia_tables(url)

    candidates = []

    for i, table in enumerate(tables):
        df = flatten_columns(table)
        cols = [c.lower() for c in df.columns]

        has_date = any("date" in c for c in cols)
        has_added = any("added" in c for c in cols)
        has_removed = any("removed" in c for c in cols)

        if has_date and has_added and has_removed:
            candidates.append((i, df))

    if not candidates:
        raise ValueError(f"No change table found for {url}")

    # Usually there is only one. If multiple, take the largest one.
    idx, change_df = max(candidates, key=lambda x: len(x[1]))

    print(f"Found change table at table index {idx}, rows={len(change_df)}")
    return change_df


def standardize_change_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize column names from Wikipedia change tables.
    Expected output columns include:
    Date, Added_Ticker, Added_Security, Removed_Ticker, Removed_Security, Reason
    """
    df = flatten_columns(df)

    rename_map = {}

    for col in df.columns:
        c = col.lower()

        if c == "date" or "date" in c:
            rename_map[col] = "Date"

        elif "added" in c and "ticker" in c:
            rename_map[col] = "Added_Ticker"
        elif "added" in c and ("security" in c or "company" in c):
            rename_map[col] = "Added_Security"

        elif "removed" in c and "ticker" in c:
            rename_map[col] = "Removed_Ticker"
        elif "removed" in c and ("security" in c or "company" in c):
            rename_map[col] = "Removed_Security"

        elif "reason" in c:
            rename_map[col] = "Reason"

    df = df.rename(columns=rename_map)

    # Keep only useful standardized columns if they exist
    useful_cols = [
        "Date",
        "Added_Ticker",
        "Added_Security",
        "Removed_Ticker",
        "Removed_Security",
        "Reason",
    ]

    existing = [c for c in useful_cols if c in df.columns]
    df = df[existing].copy()

    # Parse dates
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    return df


def get_index_changes(index_name: str, url: str, years: int = 2) -> pd.DataFrame:
    raw = find_change_table(url)
    changes = standardize_change_table(raw)
    changes["Index"] = index_name

    cutoff = pd.Timestamp.today().normalize() - pd.DateOffset(years=years)

    if "Date" in changes.columns:
        changes = changes[changes["Date"] >= cutoff].copy()

    changes = changes.sort_values("Date", ascending=False).reset_index(drop=True)
    return changes


sp500_changes = get_index_changes("SP500", SP500_URL, years=2)
sp400_changes = get_index_changes("SP400", SP400_URL, years=2)

all_changes = pd.concat([sp500_changes, sp400_changes], ignore_index=True)

print(sp500_changes.head())
print(sp400_changes.head())
print(all_changes.shape)

Found change table at table index 1, rows=395
Found change table at table index 2, rows=605
        Date Added_Ticker  Added_Security Removed_Ticker   Removed_Security  \
0 2026-05-07         VEEV   Veeva Systems           CTRA     Coterra Energy   
1 2026-04-09         CASY         Casey's           HOLX            Hologic   
2 2026-03-23          VRT          Vertiv           MTCH        Match Group   
3 2026-03-23         LITE        Lumentum            MOH  Molina Healthcare   
4 2026-03-23         COHR  Coherent Corp.             LW        Lamb Weston   

                                              Reason  Index  
0  S&P 500 constituent Devon Energy Corp. is acqu...  SP500  
1  Blackstone Inc. and TPG Inc. acquired Hologic.[8]  SP500  
2                   Market capitalization change.[9]  SP500  
3                   Market capitalization change.[9]  SP500  
4                   Market capitalization change.[9]  SP500  
        Date Added_Ticker     Added_Security Removed_Ticker R

In [8]:
all_changes

,Date,Added_Ticker,Added_Security,Removed_Ticker,Removed_Security,Reason,Index
0,2026-05-07,VEEV,Veeva Systems,CTRA,Coterra Energy,S&P 500 constituent Devon Energy Corp. is acqu...,SP500
1,2026-04-09,CASY,Casey's,HOLX,Hologic,Blackstone Inc. and TPG Inc. acquired Hologic.[8],SP500
2,2026-03-23,VRT,Vertiv,MTCH,Match Group,Market capitalization change.[9],SP500
3,2026-03-23,LITE,Lumentum,MOH,Molina Healthcare,Market capitalization change.[9],SP500
4,2026-03-23,COHR,Coherent Corp.,LW,Lamb Weston,Market capitalization change.[9],SP500
...,...,...,...,...,...,...,...
110,2024-06-24,NXT,Nextracker,GO,Grocery Outlet,Market capitalization change.[39],SP400
111,2024-06-24,ALTR,Altair Engineering,LEG,Leggett & Platt,Market capitalization change.[39],SP400
112,2024-06-24,RBA,RB Global,HTZ,Hertz,Market capitalization change.[39],SP400
113,2024-06-24,ILMN,"Illumina, Inc.",GDDY,GoDaddy,Market capitalization change.[39],SP400
